# Multimodal Text-to-SQL System with RAG and Vector Search

This notebook showcases an advanced yet intuitive **Text-to-SQL + RAG (Retrieval-Augmented Generation)** system built with **LlamaIndex**, **OpenAI**, and **Pinecone** to answer natural language questions over a structured superhero database, enriched with external context from Wikipedia.

## Data Source

The dataset used in this notebook is derived from the **mini-dev subset** of the [BIRD benchmark](https://bird-bench.github.io/), focusing on a **subset of Marvel superheroes**. The data is preprocessed and stored in a single CSV file, which contains structured attributes like name, gender, race, height, weight, and more. This structured data is then loaded into an **in-memory SQLite database** for fast SQL querying.

Additionally, semantic background knowledge is retrieved from **Wikipedia articles** using LlamaIndex's `WikipediaReader`.

![RAG Architecture](./assets/rag_arch.png)

## Notebook Flow

1. **Configuration** – Define paths, API keys, model parameters, and database settings.  
2. **Database Setup** – Create an in-memory SQLite database from a CSV containing Marvel superhero data.  
3. **Model Initialization** – Load OpenAI LLM (`gpt-4o-mini`) and HuggingFace embeddings (`mxbai-embed-large-v1`).  
4. **Vector Store Setup** – Connect to Pinecone, create/load index, and build a vector store.  
5. **Wikipedia Enrichment** – Optionally populate Pinecone with Wikipedia pages mapped to superheroes.  
6. **Query Engine Setup** – Create:  
   - SQL query engine (`NLSQLTableQueryEngine`) for structured questions  
   - Vector query engine (`RetrieverQueryEngine`) for semantic/Wikipedia-based questions  
7. **Combined Engine** – Wrap both engines into a unified `SQLAutoVectorQueryEngine` with tool selection.  
8. **Query Execution** – Define helper to run natural language queries.  
9. **System Testing** – Run example multimodal queries and evaluate structured + semantic responses.

## Architecture Components

- **SQL Database**: SQLite with comprehensive Marvel superhero dataset
- **Embeddings**: MixedBread AI `mxbai-embed-large-v1` for high-quality vector representations used for vector search
- **LLM**: OpenAI GPT-4o-mini for query understanding and response generation; alternatively, you can use HuggingFace models.
- **Vector Storage**: Pinecone serverless vector database for scalable similarity search
- **SQL Database**: SQLite with comprehensive Marvel superhero dataset

## Database Schema
The system operates on a rich superhero dataset with the following attributes:

- `id`: Unique identifier for each superhero
- `superhero_name`: Common name or alias (e.g., "Spider-Man")
- `full_name`: Real name or identity (e.g., "Peter Parker")
- `gender`: Gender of the character
- `eye_colour`: Eye color
- `hair_colour`: Hair color
- `skin_colour`: Skin color
- `race`: Species or race (e.g., Human, Mutant, Alien)
- `publisher_name`: Publishing entity (e.g., Marvel Comics)
- `alignment`: Moral alignment (Good, Bad, Neutral)
- `height_cm`: Height in centimeters
- `weight_kg`: Weight in kilograms

## Table of Contents

1. [Step 1: Import Required Libraries](#step-1-import-required-libraries)  
2. [Step 2: Configuration Setup](#step-2-configuration-setup)
3. [Step 3: Database Creation and Data Loading](#step-3-database-creation-and-data-loading)
4. [Step 4: Model Initialization](#step-4-model-initialization)
5. [Step 5: Pinecone Vector Store Setup](#step-5-pinecone-vector-store-setup)
6. [Step 6: Wikipedia Data Integration](#step-6-wikipedia-data-integration)
7. [Step 7: Query Engine Setup](#step-7-query-engine-setup)
8. [Step 8: Combined Multimodal Query Engine](#step-8-combined-multimodal-query-engine)
9. [Step 9: Run Example Queries](#step-9-run-example-queries)

## Step 1: Import Required Libraries

In [ ]:
# Standard library imports
import json
import os
from typing import Any, Dict, List

# Data processing
import pandas as pd

# LlamaIndex core components
from llama_index.core import (
    Settings,
    SQLDatabase,
    StorageContext,
    VectorStoreIndex,
)

# LlamaIndex specialized components
# LlamaIndex query engines
from llama_index.core.query_engine import (
    NLSQLTableQueryEngine,
    RetrieverQueryEngine,
    SQLAutoVectorQueryEngine,
)
from llama_index.core.retrievers import VectorIndexAutoRetriever
from llama_index.core.tools import QueryEngineTool
from llama_index.core.vector_stores import MetadataInfo, VectorStoreInfo

# Hugging Face imports
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_index.readers.wikipedia import WikipediaReader

# LlamaIndex vector stores and readers
from llama_index.vector_stores.pinecone import PineconeVectorStore

# Third-party imports for vector storage
from pinecone import Pinecone, ServerlessSpec

# Database operations
from sqlalchemy import (
    Column,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    create_engine,
    insert,
)


print("All libraries imported successfully!")

## Step 2: Configuration Setup

Let's configure our system with all the necessary settings for:

- **Database paths**: Location of our superhero CSV data and Wikipedia title mappings
- **Pinecone settings**: API credentials and index configuration for vector storage
- **Model settings**: LLM and embedding model specifications
- **System parameters**: Temperature, token limits, and other operational settings


In [ ]:
BASE_DIR = "/projects/aieng/multimodal_bootcamp/table_processing"
MARVEL_DATA_PATH = os.path.join(
    BASE_DIR, "data", "bird_minidev", "marvel_superhero_filtered.csv"
)
WIKI_TITLES_PATH = os.path.join(
    BASE_DIR, "data", "bird_minidev", "marvel_superhero_wiki_titles.json"
)

⚠️ **Note:** If you want to load an **arbitrary model**, make sure to **edit the model cache directory** (e.g., point it to your **home directory**) so you have proper write access.

In [ ]:
MODEL_CACHE_DIR = os.path.join(BASE_DIR, "model_weights")
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE_DIR
os.environ["HF_HOME"] = MODEL_CACHE_DIR

In [ ]:
def load_configuration():
    """Load and return configuration settings for the multimodal system."""
    config = {
        # Database settings
        "db_path": ":memory:",  # In-memory SQLite database
        "csv_path": MARVEL_DATA_PATH,
        "wiki_titles_path": WIKI_TITLES_PATH,
        # Pinecone settings for vector storage
        "pinecone_api_key": os.getenv("PINECONE_API_KEY"),
        "pinecone_environment": "us-west1-gcp",
        "pinecone_index_name": "multimodal-text2sql",
        "pinecone_namespace": "superheroes",
        # Model settings
        "use_openai": True,
        "llm_model": "gpt-4o-mini",  # Model name
        "embedding_model": "mixedbread-ai/mxbai-embed-large-v1",
        "temperature": 0.1,
        # Table settings
        "table_name": "marvel_superheroes",
    }

    print("Configuration loaded successfully!")
    return config


# Load the configuration
config = load_configuration()

## Step 3: Database Creation and Data Loading

Now we'll create our SQLite database and populate it with Marvel superhero data. This involves:

1. **Database Schema Definition**: Creating a comprehensive table structure with 12 columns
2. **Data Cleaning**: Handling missing values and ensuring data type consistency
3. **Data Loading**: Inserting all superhero records into the database
4. **Statistics Display**: Showing data distribution and characteristics

The database serves as the structured foundation for our SQL queries.


In [ ]:
def create_database(csv_path: str, table_name: str):
    """Create an in-memory SQLite database with Marvel superhero data."""
    print("Creating Marvel superhero database...")

    # Create an in-memory SQLite database
    engine = create_engine("sqlite:///:memory:")
    metadata_obj = MetaData()

    # Define the comprehensive table structure
    table = Table(
        table_name,
        metadata_obj,
        Column("id", Integer, primary_key=True),
        Column("superhero_name", String(50), nullable=False),
        Column("full_name", String(100)),
        Column("gender", String(10)),
        Column("eye_colour", String(20)),
        Column("hair_colour", String(20)),
        Column("skin_colour", String(20)),
        Column("race", String(30)),
        Column("publisher_name", String(30)),
        Column("alignment", String(15)),  # Good, Bad, Neutral
        Column("height_cm", Float),
        Column("weight_kg", Float),
    )

    # Create all tables in the database
    metadata_obj.create_all(engine)
    print("Database schema created")

    try:
        # Read and clean the CSV data
        print("Loading and cleaning data...")
        df = pd.read_csv(csv_path)

        # Handle missing values
        df = df.fillna("Unknown")

        # Convert numeric columns safely
        df["height_cm"] = pd.to_numeric(df["height_cm"], errors="coerce")
        df["weight_kg"] = pd.to_numeric(df["weight_kg"], errors="coerce")

        # Insert data into the database
        sample_data = df.to_dict("records")
        for row in sample_data:
            stmt = insert(table).values(**row)
            with engine.begin() as connection:
                connection.execute(stmt)

        # Display comprehensive statistics
        print(f"Successfully loaded {len(sample_data)} superhero records")
        print("Data Statistics:")
        print(
            f"• Alignments: {df['alignment'].nunique()} unique ({', '.join(df['alignment'].value_counts().head(3).index.tolist())})"
        )
        print(
            f"• Races: {df['race'].nunique()} unique (top: {', '.join(df['race'].value_counts().head(3).index.tolist())})"
        )
        print(f"• Publishers: {df['publisher_name'].nunique()} unique")
        print(f"• Gender distribution: {dict(df['gender'].value_counts())}")

    except FileNotFoundError:
        print(f"CSV file not found: {csv_path}")
        return None, None
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        return None, None

    return engine, table

In [ ]:
# Create the database with our superhero data
engine, table = create_database(config["csv_path"], config["table_name"])

## Step 4: Model Initialization

Let's initialize our AI models that power the intelligent querying system:

1. **Embedding Model**: MixedBread AI's `mxbai-embed-large-v1` for high-quality vector representations
2. **Language Model**: OpenAI's GPT-4o-mini for natural language understanding and SQL generation
3. **Global Settings**: Configure LlamaIndex to use our chosen models

These models work together to understand natural language queries and generate appropriate responses.


In [ ]:
def setup_models(config: Dict[str, Any]):
    """Initialize and configure LLM and embedding models."""
    print("Initializing AI models...")

    # Initialize embedding model for vector representations
    print(f"Loading embedding model: {config['embedding_model']}")
    embedding_model = HuggingFaceEmbedding(model_name=config["embedding_model"])

    print(f"Configuring LLM: {config['llm_model']}")
    if config["use_openai"]:
        # Initialize OpenAI LLM for query understanding and generation
        llm = OpenAI(
            model=config["llm_model"],
            temperature=config["temperature"],
        )
    else:
        llm = HuggingFaceLLM(
            model_name=config["llm_model"],
            tokenizer_name=config["llm_model"],
            context_window=2048,
            max_new_tokens=256,
            generate_kwargs={"temperature": config["temperature"], "do_sample": True},
        )

    # Set global settings for LlamaIndex
    Settings.llm = llm
    Settings.embed_model = embedding_model

    print("Models initialized successfully!")
    print(f"Temperature: {config['temperature']} (for consistent responses)")

    return llm, embedding_model

In [ ]:
# Initialize our AI models
if engine:
    llm, embedding_model = setup_models(config)
    print("Models are ready!")
else:
    print("Skipping model setup - database not available")

## Step 5: Pinecone Vector Store Setup

Now we'll set up our Pinecone vector database for semantic search capabilities:

1. **Pinecone Client Initialization**: Connect to Pinecone cloud service
2. **Index Creation**: Create or connect to a vector index with proper dimensions
3. **Vector Store Configuration**: Set up the LlamaIndex-Pinecone integration
4. **Storage Context**: Prepare the storage context for vector operations

This vector store will enable semantic similarity search over our superhero data and Wikipedia content.


In [ ]:
def setup_vector_store(config: Dict[str, Any]):
    """Initialize Pinecone vector store and index."""
    try:
        # Initialize Pinecone client
        print("Connecting to Pinecone...")
        pc = Pinecone(api_key=config["pinecone_api_key"])

        # Check if index exists, create if not
        existing_indexes = [idx.name for idx in pc.list_indexes()]

        if config["pinecone_index_name"] not in existing_indexes:
            print(f"Creating new Pinecone index: {config['pinecone_index_name']}")
            pc.create_index(
                name=config["pinecone_index_name"],
                dimension=1024,  # MixedBread AI embedding dimension
                metric="cosine",  # Cosine similarity for semantic search
                spec=ServerlessSpec(cloud="aws", region="us-west-2"),
            )
            print("Index created, waiting for initialization...")
        else:
            print(f"Using existing index: {config['pinecone_index_name']}")

        # Get Pinecone index and create vector store
        pinecone_index = pc.Index(config["pinecone_index_name"])
        vector_store = PineconeVectorStore(
            pinecone_index=pinecone_index, namespace=config["pinecone_namespace"]
        )

        # Create storage context and vector index
        storage_context = StorageContext.from_defaults(vector_store=vector_store)
        vector_index = VectorStoreIndex([], storage_context=storage_context)

        print("Pinecone vector store ready!")
        print(f"Using namespace: {config['pinecone_namespace']}")

        return vector_index

    except Exception as e:
        print(f"Failed to setup vector store: {e}")
        raise

In [ ]:
# Setup vector store
vector_index = setup_vector_store(config)
print("Vector search capabilities activated!")

## Step 6: Wikipedia Data Integration

This step populates our vector index with rich Wikipedia content about superheroes. This process involves:

1. **Data Extraction**: Getting superhero names from our database
2. **Wikipedia Mapping**: Using our pre-built title mappings for accurate Wikipedia searches
3. **Content Loading**: Fetching full Wikipedia articles using LlamaIndex's WikipediaReader
4. **Vector Indexing**: Converting articles to embeddings and storing in Pinecone

In [ ]:
def populate_vector_index(
    vector_index, engine, table_name: str, config: Dict[str, Any]
):
    """Populate vector index with Wikipedia data for superheroes in database."""
    print("Loading Wikipedia data for superheroes...")

    try:
        # Get superhero names from database
        df = pd.read_sql_table(table_name, engine)
        names = df["superhero_name"].dropna().unique().tolist()
        print(f"Found {len(names)} unique superhero names")

        # Load wiki titles mapping
        if not os.path.exists(config["wiki_titles_path"]):
            print(f"Wiki titles file not found: {config['wiki_titles_path']}")
            print(
                "You can create this file by mapping superhero names to Wikipedia titles"
            )
            return vector_index

        with open(config["wiki_titles_path"], "r") as f:
            wiki_titles = json.load(f)

        # Load Wikipedia data
        wiki_reader = WikipediaReader()
        all_docs = []
        success, failed = 0, 0

        print("Fetching Wikipedia articles...")
        for i, name in enumerate(names[:10], 1):  # Limit to first 10 for demo
            try:
                if name not in wiki_titles:
                    print(f"No wiki title mapping for: {name}")
                    failed += 1
                    continue

                wiki_title = wiki_titles[name]
                print(f"   [{i}/10] Loading: '{wiki_title}' for '{name}'")

                # Load Wikipedia page
                wiki_docs = wiki_reader.load_data(
                    pages=[wiki_title], auto_suggest=False
                )

                for doc in wiki_docs:
                    doc.metadata = {
                        "title": wiki_title,
                        "original_name": name,
                        "source": "wikipedia",
                    }
                    all_docs.append(doc)
                success += 1

            except Exception as e:
                print(f"Failed to load '{name}': {e}")
                failed += 1

        print(f"Results: {success} successful, {failed} failed")

        # Index documents
        if all_docs:
            print(f"Indexing {len(all_docs)} documents into Pinecone...")
            storage_context = vector_index.storage_context
            vector_index = VectorStoreIndex.from_documents(
                all_docs, storage_context=storage_context
            )
            print("Vector index populated successfully!")

        return vector_index

    except Exception as e:
        print(f"Failed to populate vector index: {e}")
        raise

In [ ]:
vector_index = populate_vector_index(vector_index, engine, config["table_name"], config)

## Step 7: Query Engine Setup

Now we'll create the specialized query engines that form the core of our multimodal system:

### 1. SQL Query Engine
- **Purpose**: Handles structured queries that can be answered with SQL
- **Capabilities**: Counts, comparisons, filters, sorting, numerical analysis
- **Technology**: NLSQLTableQueryEngine with natural language to SQL translation

### 2. Vector Query Engine  
- **Purpose**: Handles semantic queries requiring contextual understanding
- **Capabilities**: Stories, origins, relationships, biographical information
- **Technology**: VectorIndexAutoRetriever with similarity search

Both engines work together to provide comprehensive query capabilities.


In [ ]:
def setup_query_engines(sql_database, vector_index, table_name: str, llm):
    """Set up SQL and vector query engines."""
    try:
        # Create SQL query engine for structured data queries
        print("Creating SQL query engine...")
        sql_query_engine = NLSQLTableQueryEngine(
            sql_database=sql_database, tables=[table_name], llm=llm
        )
        print("SQL query engine ready for structured queries")

        # Create vector store info for auto-retrieval
        print("Configuring vector query engine...")
        vector_store_info = VectorStoreInfo(
            content_info="Wikipedia articles about superheroes, their powers, origins, and stories",
            metadata_info=[
                MetadataInfo(
                    name="title", type="str", description="The name of the superhero"
                ),
                MetadataInfo(
                    name="source",
                    type="str",
                    description="The source of the information (wikipedia)",
                ),
            ],
        )

        # Create vector auto-retriever and query engine
        vector_auto_retriever = VectorIndexAutoRetriever(
            vector_index,
            vector_store_info=vector_store_info,
            similarity_top_k=3,  # Return top 3 most similar results
        )

        vector_query_engine = RetrieverQueryEngine.from_args(
            vector_auto_retriever, llm=llm
        )
        print("Vector query engine ready for semantic queries")

        return sql_query_engine, vector_query_engine

    except Exception as e:
        print(f"Failed to setup query engines: {e}")
        raise

In [ ]:
# Create SQL database object for LlamaIndex
sql_database = SQLDatabase(engine)

# Setup the query engines
sql_query_engine, vector_query_engine = setup_query_engines(
    sql_database, vector_index, config["table_name"], llm
)

## Step 8: Combined Multimodal Query Engine

This is where the magic happens! We'll create a unified query engine that intelligently routes queries to the appropriate specialized engine:

### Intelligent Query Routing
The system automatically determines whether a query should be handled by:
- **SQL Engine**: For analytical queries (counts, comparisons, filters)
- **Vector Engine**: For semantic queries (stories, context, relationships)

### Tool Integration
We'll create QueryEngineTool objects that wrap our specialized engines and provide clear descriptions for the LLM to understand when to use each tool.

### SQLAutoVectorQueryEngine
This combines both tools into a single interface that can handle any type of query seamlessly.


In [ ]:
def setup_combined_query_engine(sql_query_engine, vector_query_engine, llm):
    """Set up the combined multimodal query engine with intelligent routing."""
    try:
        # Create SQL tool with detailed description for the LLM
        print("Creating SQL query tool...")
        sql_tool = QueryEngineTool.from_defaults(
            query_engine=sql_query_engine,
            description=(
                "Use this tool to answer questions involving structured data analysis such as:\n"
                "• Counts and statistics (how many, total number, average)\n"
                "• Comparisons and rankings (tallest, heaviest, most common)\n"
                "• Filters and searches (find all characters with specific attributes)\n"
                "• Numerical analysis (height, weight, measurements)\n"
                "• Categorical data (publishers, gender, race, alignments)\n"
                "• Sorting and ordering (sort by height, group by alignment)\n"
                "Ideal for analytical queries that can be answered using SQL operations."
            ),
        )

        # Create vector tool with detailed description for the LLM
        print("Creating vector query tool...")
        vector_tool = QueryEngineTool.from_defaults(
            query_engine=vector_query_engine,
            description=(
                "Use this tool to answer semantic questions about superheroes using "
                "rich contextual information from Wikipedia including:\n"
                "• Character origins and backstories\n"
                "• Powers and abilities descriptions\n"
                "• Story arcs and plot details\n"
                "• Relationships and team affiliations\n"
                "• Cultural significance and impact\n"
                "• Biographical and historical information\n"
                "Best for questions requiring deep contextual understanding."
            ),
        )

        # Create the combined query engine
        print("Integrating tools into unified engine...")
        combined_query_engine = SQLAutoVectorQueryEngine(
            sql_tool,  # Tool for structured queries
            vector_tool,  # Tool for semantic queries
            llm=llm,  # LLM for intelligent routing
        )

        print("Combined query engine ready!")

        return combined_query_engine

    except Exception as e:
        print(f"Failed to setup combined query engine: {e}")
        raise

In [ ]:
# Create the combined query engine
combined_query_engine = setup_combined_query_engine(
    sql_query_engine, vector_query_engine, llm
)

## Step 9: Run Example Queries

Now let's test our system with a variety of queries to demonstrate its capabilities:


In [ ]:
def execute_query(query_engine, question: str) -> str:
    """Execute a natural language query using the combined engine."""
    if not query_engine:
        return "System not initialized properly."

    try:
        print(f"Processing query: '{question}'")
        print("Analyzing query type and routing to appropriate engine...")

        response = query_engine.query(question)
        return str(response)
    except Exception as e:
        error_msg = f"Query execution failed: {str(e)}"
        print(error_msg)
        return error_msg

In [ ]:
def test_queries(query_engine, queries: List[str]):
    """Test a set of queries and display results."""
    for i, query in enumerate(queries, 1):
        print(f"\n[{i}/{len(queries)}] Query: {query}")

        try:
            response = execute_query(query_engine, query)
            print(f"Response: {response}")

        except Exception as e:
            print(f"Error: {str(e)}")

        print("-" * 70)

In [ ]:
# Test analytical/SQL queries
queries = [
    "Who has the highest weight in the database? What color is their eyes?",
    "Who is the tallest superhero? Tell me about this character.",
    "Which superheroes have Red hair in the database? What is their origin story?",
]

test_queries(combined_query_engine, queries)